# BIND2 Redshift — multi-z field check & redshift-dependence

Validation + exploration notebook for the **`--condition_redshift`** model
(`fm_redshift`): a single flow-matching emulator trained on the multi-redshift
dataset (`train_data_multiz_128_cpu`, 7 snapshots z≈0→3) that paints mass
(`DM_hydro, Gas, Stars`) **and** the 4 gas-thermo fields
(`compton_y, T, entropy, P_e`), conditioned on the scale factor a=1/(1+z).

Structure (parallels `analysis_thermo.ipynb`, then adds the z-axis):
1. Visual check across redshift
2. Per-redshift fidelity scorecard (mass + thermo)
3. **Redshift evolution of field amplitudes** — truth vs BIND
4. **Y–M relation evolution with z**
5. **Conditioning response** — fix structure+params, sweep the conditioning a
6. Numeric summary

⚠ The z>0 thermo *physics* (comoving→physical a-factors in the data generator)
is not independently validated — treat absolute high-z thermo amplitudes with
care. The emulator-fidelity checks (BIND vs the truth maps it was trained on)
are unaffected by that caveat.

In [ ]:
import sys, os
sys.path.insert(0, '/mnt/home/mlee1/vdm_bind2')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import re
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from collections import defaultdict
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from bind.data import (load_file_list, AstroDataset, NormStats, THERMO_KEYS,
                       N_THERMO, SNAPSHOT_REDSHIFTS, z_to_a, a_to_z)
from bind.train import FlowMatchingLit
from bind.metrics import CHANNEL_NAMES
from bind.inference.pipeline import _denormalize_to_physical

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

MASS_NAMES   = list(CHANNEL_NAMES)          # ['DM_hydro', 'Gas', 'Stars']
THERMO_NAMES = list(THERMO_KEYS)            # compton_y, temperature, entropy, pressure
THERMO_UNITS = ['dimensionless', 'K', 'keV cm$^2$', 'Pa']
ALL_NAMES    = MASS_NAMES + THERMO_NAMES    # physical channel order (7)
print(f'Device: {device} | {N_THERMO} thermo channels: {THERMO_NAMES}')
print('snapshot -> z:', SNAPSHOT_REDSHIFTS)

In [ ]:
# ── Configuration ────────────────────────────────
DATA_ROOT = '/mnt/home/mlee1/ceph/train_data_multiz_128_cpu'   # multi-z dataset
RUNS_DIR  = Path('/mnt/home/mlee1/ceph/fm_runs')
RUN_NAME  = 'fm_redshift'      # mass + thermo + scale-factor conditioning (out_ch = 8)
RUN_DIR   = RUNS_DIR / RUN_NAME
CKPT_NAME = 'last.ckpt'        # tracks the live run; or 'epochNNN-....ckpt'

BOX_SIZE  = 6.25               # Mpc/h per halo cutout
N_PIX     = 128

N_PER_Z    = 256               # test patches per redshift bin
N_STEPS    = 20                # ODE steps for FM sampling
BATCH_SIZE = 64
N_WORKERS  = 8
SEED       = 42

# Multi-z test set: cache enumerates train/sim_i/snap_j/...; recursive=True reads
# file_list_cache_multiz.txt (pre-built). Group by snapshot (=redshift) so we can
# draw a balanced sample per z and keep file order == array order.
rng = np.random.RandomState(SEED)
all_test = load_file_list(DATA_ROOT, 'test', recursive=True)
by_snap = defaultdict(list)
for f in all_test:
    m = re.search(r'snap_?(\d+)', f)
    if m:
        by_snap[int(m.group(1))].append(f)
snaps = sorted(by_snap, reverse=True)        # high snap = low z first
print(f'{len(all_test)} test patches across snaps: '
      + ', '.join(f'{s}(z={SNAPSHOT_REDSHIFTS.get(s, np.nan):.2f}):{len(by_snap[s])}'
                  for s in snaps))

test_files, halo_mass, zarr = [], [], []
for s in snaps:
    fs = by_snap[s]
    pick = rng.permutation(len(fs))[:N_PER_Z]
    for i in pick:
        d = np.load(fs[i])
        if not all(k in d.files for k in THERMO_KEYS):
            continue
        test_files.append(fs[i])
        halo_mass.append(float(d['halo_mass']) if 'halo_mass' in d.files else np.nan)
        zarr.append(float(d['redshift']) if 'redshift' in d.files
                    else SNAPSHOT_REDSHIFTS[s])
halo_mass = np.asarray(halo_mass)
zarr = np.asarray(zarr)
# Round to the nominal snapshot redshifts so binning is exact.
z_levels = np.array(sorted(set(np.round(zarr, 3))))
print(f'\nSelected {len(test_files)} patches | z levels: {z_levels}')
for zl in z_levels:
    print(f'  z={zl:.3f}: {(np.round(zarr,3)==zl).sum()} patches')

In [ ]:
# ── Model loading & inference (scale-factor conditioned) ──
def _ckpt_epoch(ckpt):
    m = re.search(r'epoch(\d+)', Path(ckpt).name)
    if m:
        return int(m.group(1))
    try:
        return torch.load(ckpt, map_location='cpu', weights_only=False).get('epoch', '?')
    except Exception:
        return '?'


def load_run(run_dir, ckpt_name='last.ckpt'):
    run_dir = Path(run_dir)
    ckpt = run_dir / 'checkpoints' / ckpt_name
    if not ckpt.exists():
        raise FileNotFoundError(f'Missing checkpoint: {ckpt}')
    model = FlowMatchingLit.load_from_checkpoint(str(ckpt), map_location=device)
    model.eval().to(device)
    ns = NormStats.load(run_dir / 'norm_stats.npz')
    assert ns.predict_thermo, 'norm_stats has no thermo stats — wrong run?'
    assert getattr(model.hparams, 'condition_redshift', False), \
        'model is NOT redshift-conditioned — wrong run?'
    return model, ns, _ckpt_epoch(ckpt)


@torch.no_grad()
def generate_predictions(model, ns, files, batch_size=64, n_steps=20, n_workers=8):
    """Sample DMO->hydro conditioned on each patch's scale factor a=1/(1+z).

    Returns physical-space (real, gen) each (N, 3 + N_THERMO, H, W) plus the
    per-patch scale_factor array.
    """
    ds = AstroDataset(files, ns, condition_redshift=True)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=n_workers, pin_memory=True,
                        persistent_workers=(n_workers > 0))
    real, gen, sf = [], [], []
    for batch in tqdm(loader, desc='sampling', leave=False):
        cond   = batch['condition'].to(device)
        ls     = batch['large_scale'].to(device)
        params = batch['params'].to(device)
        a      = batch['scale_factor'].to(device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            g = model.fm.sample(cond, ls, params, n_steps=n_steps, scale_factor=a)
        real.append(_denormalize_to_physical(batch['target'].numpy().copy(), ns))
        gen.append(_denormalize_to_physical(g.float().cpu().numpy(), ns))
        sf.append(batch['scale_factor'].numpy())
    return np.concatenate(real), np.concatenate(gen), np.concatenate(sf)

In [ ]:
model, ns, epoch = load_run(RUN_DIR, CKPT_NAME)
print(f'Loaded {RUN_NAME} / {CKPT_NAME} (epoch {epoch}) | '
      f'stars_two_head={ns.stars_two_head}, predict_thermo={ns.predict_thermo}, '
      f'condition_redshift={getattr(model.hparams, "condition_redshift", False)}')

real, gen, sf = generate_predictions(
    model, ns, test_files, batch_size=BATCH_SIZE, n_steps=N_STEPS, n_workers=N_WORKERS)
z_pred = a_to_z(sf)                                   # redshift per patch (from a)

assert real.shape[1] == 3 + N_THERMO, real.shape
assert len(real) == len(halo_mass) == len(zarr)
mass_real,   mass_gen   = real[:, :3], gen[:, :3]
thermo_real, thermo_gen = real[:, 3:], gen[:, 3:]
# Round redshift labels to the nominal snapshot grid for clean per-z binning.
zlab = np.round(zarr, 3)
print(f'{len(real)} patches | channels = {real.shape[1]} ({ALL_NAMES})')
print('per-z patch counts:', {float(zl): int((zlab == zl).sum()) for zl in z_levels})
# Keep the model around for the conditioning-sweep section (§5).

## 1. Visual check across redshift

One representative high-mass halo per redshift level: truth vs BIND for Gas and
two thermo fields. Amplitudes should fall toward higher z and BIND should track
truth at each z.

In [ ]:
show_fields = [('Gas', 1, False), ('compton_y', 3, True), ('temperature', 4, True)]
nz = len(z_levels)
fig, axes = plt.subplots(len(show_fields) * 2, nz, figsize=(2.0 * nz, 2.0 * len(show_fields) * 2))
for zi, zl in enumerate(z_levels):
    idx_z = np.where(zlab == zl)[0]
    pick = idx_z[np.argmax(halo_mass[idx_z])]        # most massive halo at this z
    for fi, (nm, ch, is_thermo) in enumerate(show_fields):
        r = real[pick, ch]; g = gen[pick, ch]
        vmax = np.log10(np.nanmax(r) + 1e-30)
        vmin = vmax - 4
        for k, (img, tag) in enumerate([(r, 'truth'), (g, 'BIND')]):
            ax = axes[2 * fi + k, zi]
            ax.imshow(np.log10(img + 1e-30), vmin=vmin, vmax=vmax, cmap='magma')
            ax.set_xticks([]); ax.set_yticks([])
            if zi == 0:
                ax.set_ylabel(f'{nm}\n{tag}', fontsize=8)
            if fi == 0 and k == 0:
                ax.set_title(f'z={zl:.2f}\nlogM={np.log10(halo_mass[pick]):.1f}', fontsize=8)
fig.suptitle('Most-massive halo per redshift — log10 fields (truth / BIND)', y=1.005)
plt.tight_layout(); plt.show()

## 2. Per-redshift fidelity scorecard

For each redshift level: median |relative| total-mass error (mass channels) and
median per-pixel |Δlog10| (dex) error for thermo channels. Watch for fidelity
degrading at high z, where halos are rarer and the training set is thinner.

In [ ]:
def _rel_mass_err(r, g):
    mr = r.sum(axis=(-2, -1)); mg = g.sum(axis=(-2, -1))
    return np.abs(mg - mr) / (np.abs(mr) + 1e-30)

def _dex_err(r, g, floor):
    m = (r > floor) & (g > floor)
    return np.abs(np.log10(g[m] + 1e-30) - np.log10(r[m] + 1e-30))

print(f'{"z":>6} {"N":>5} | ' + ' '.join(f'{n[:7]:>8}' for n in MASS_NAMES)
      + ' || ' + ' '.join(f'{n[:7]:>8}' for n in THERMO_NAMES))
print('-' * 96)
score = {}
for zl in z_levels:
    idx = np.where(zlab == zl)[0]
    row_mass = [np.median(_rel_mass_err(mass_real[idx, c], mass_gen[idx, c])) * 100
                for c in range(3)]
    row_th = []
    for j in range(N_THERMO):
        fl = np.percentile(thermo_real[idx, j][thermo_real[idx, j] > 0], 50) * 1e-3
        row_th.append(np.median(_dex_err(thermo_real[idx, j], thermo_gen[idx, j], fl)))
    score[zl] = (row_mass, row_th)
    print(f'{zl:6.2f} {len(idx):5d} | '
          + ' '.join(f'{v:7.1f}%' for v in row_mass)
          + ' || ' + ' '.join(f'{v:8.3f}' for v in row_th))
print('\nmass = median |rel total-mass| error (%); thermo = median |Δlog10| (dex)')

# Plot the trends.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for c, nm in enumerate(MASS_NAMES):
    axes[0].plot(z_levels, [score[zl][0][c] for zl in z_levels], 'o-', label=nm)
axes[0].set_xlabel('z'); axes[0].set_ylabel('median |rel mass err| (%)')
axes[0].set_title('Mass-channel fidelity vs z'); axes[0].legend(); axes[0].grid(alpha=.3)
for j, nm in enumerate(THERMO_NAMES):
    axes[1].plot(z_levels, [score[zl][1][j] for zl in z_levels], 's-', label=nm)
axes[1].set_xlabel('z'); axes[1].set_ylabel('median |Δlog10| (dex)')
axes[1].set_title('Thermo-channel fidelity vs z'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3. Redshift evolution of field amplitudes — truth vs BIND

The core redshift-dependence plot: median per-patch field amplitude as a function
of z, for truth and BIND overlaid. A redshift-aware emulator must reproduce the
*evolution*, not just the z=0 normalization. Mass channels use total patch mass;
thermo channels use the mean over positive pixels.

In [ ]:
def _patch_amp(arr, thermo):
    # thermo: mean over positive pixels; mass: total patch mass.
    if thermo:
        out = np.full(len(arr), np.nan)
        for i, m in enumerate(arr):
            pos = m[m > 0]
            if pos.size:
                out[i] = pos.mean()
        return out
    return arr.sum(axis=(-2, -1))

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
panels = [('Gas', mass_real[:, 1], mass_gen[:, 1], False),
          ('Stars', mass_real[:, 2], mass_gen[:, 2], False),
          ('DM_hydro', mass_real[:, 0], mass_gen[:, 0], False),
          ('compton_y', thermo_real[:, 0], thermo_gen[:, 0], True),
          ('temperature', thermo_real[:, 1], thermo_gen[:, 1], True),
          ('entropy', thermo_real[:, 2], thermo_gen[:, 2], True),
          ('pressure', thermo_real[:, 3], thermo_gen[:, 3], True)]
for ax, (nm, r, g, th) in zip(axes.flat, panels):
    ar, ag = _patch_amp(r, th), _patch_amp(g, th)
    mr = np.array([np.nanmedian(ar[zlab == zl]) for zl in z_levels])
    mg = np.array([np.nanmedian(ag[zlab == zl]) for zl in z_levels])
    lo = np.array([np.nanpercentile(ar[zlab == zl], 16) for zl in z_levels])
    hi = np.array([np.nanpercentile(ar[zlab == zl], 84) for zl in z_levels])
    ax.fill_between(z_levels, lo, hi, alpha=0.15, color='k')
    ax.plot(z_levels, mr, 'ko-', label='truth', lw=2)
    ax.plot(z_levels, mg, 'C3s--', label='BIND', lw=2)
    ax.set_yscale('log'); ax.set_xlabel('z')
    ax.set_ylabel('mean (pos px)' if th else 'total patch mass')
    ax.set_title(nm); ax.grid(alpha=.3); ax.legend(fontsize=9)
axes.flat[-1].axis('off')
fig.suptitle('Redshift evolution of field amplitudes — truth (black) vs BIND (red)', y=1.01)
plt.tight_layout(); plt.show()

## 4. Y–M relation evolution with redshift

The integrated Compton-Y vs halo mass relation, fit per redshift level. Both
the amplitude and the truth/BIND agreement are tracked vs z — the headline
multi-z thermo science check.

In [ ]:
iy = THERMO_NAMES.index('compton_y')
logM = np.log10(halo_mass)
Y_real = thermo_real[:, iy].sum(axis=(-2, -1))       # integrated Y (pixel sum)
Y_gen  = thermo_gen[:, iy].sum(axis=(-2, -1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.viridis(np.linspace(0, 1, len(z_levels)))
slopes = {}
for zl, col in zip(z_levels, cmap):
    idx = np.where((zlab == zl) & (Y_real > 0) & np.isfinite(logM))[0]
    if len(idx) < 8:
        continue
    x = logM[idx]
    # truth + BIND fits
    for Y, mk, label in [(Y_real, 'o', 'truth'), (Y_gen, 'x', 'BIND')]:
        yv = np.log10(Y[idx] + 1e-30)
        b, a0 = np.polyfit(x, yv, 1)
        xg = np.linspace(x.min(), x.max(), 20)
        axes[0].plot(xg, a0 + b * xg, ls=('-' if label == 'truth' else '--'),
                     color=col, lw=1.6, label=f'z={zl:.2f} {label} (β={b:.2f})')
        slopes.setdefault(zl, {})[label] = b
axes[0].set_xlabel('log10 M200 [Msun/h]'); axes[0].set_ylabel('log10 Y_int')
axes[0].set_title('Y–M relation per redshift'); axes[0].grid(alpha=.3)
axes[0].legend(fontsize=6, ncol=2)

# Amplitude at fixed mass (logM=13.5) vs z, truth vs BIND.
pivot = 13.5
amp_t, amp_b = [], []
for zl in z_levels:
    idx = np.where((zlab == zl) & (Y_real > 0) & np.isfinite(logM))[0]
    if len(idx) < 8:
        amp_t.append(np.nan); amp_b.append(np.nan); continue
    x = logM[idx]
    bt, at = np.polyfit(x, np.log10(Y_real[idx] + 1e-30), 1)
    bb, ab = np.polyfit(x, np.log10(Y_gen[idx] + 1e-30), 1)
    amp_t.append(at + bt * pivot); amp_b.append(ab + bb * pivot)
axes[1].plot(z_levels, amp_t, 'ko-', label='truth', lw=2)
axes[1].plot(z_levels, amp_b, 'C3s--', label='BIND', lw=2)
axes[1].set_xlabel('z'); axes[1].set_ylabel(f'log10 Y_int at logM={pivot}')
axes[1].set_title('Y(M=pivot) evolution'); axes[1].grid(alpha=.3); axes[1].legend()
plt.tight_layout(); plt.show()
print('Y–M slopes (truth / BIND) per z:',
      {float(zl): (round(v.get('truth', np.nan), 2), round(v.get('BIND', np.nan), 2))
       for zl, v in slopes.items()})

## 5. Conditioning response — sweep the conditioning scale factor

A direct test that the redshift conditioning is *active*: take a fixed set of
test patches (their DMO condition, large-scale context and params held constant)
and re-paint them at **every** training scale factor a, plus interpolated values.
Because only the conditioning a changes, any variation in the output isolates what
the model learned about redshift. We expect monotonic suppression of gas/thermo
amplitudes toward high z, smoothly interpolating between the training redshifts.

In [ ]:
@torch.no_grad()
def repaint_fixed_structure(model, ns, files, a_grid, n_per=64, n_steps=20, seed=0):
    """Paint a fixed batch of patches at each scale factor in a_grid.

    Returns dict a -> physical (n_per, 3+N_THERMO, H, W). The DMO condition,
    large_scale and params are identical across a; only the conditioning a moves.
    """
    ds = AstroDataset(files[:n_per], ns, condition_redshift=True)
    loader = DataLoader(ds, batch_size=n_per, shuffle=False, num_workers=4)
    batch = next(iter(loader))
    cond = batch['condition'].to(device); ls = batch['large_scale'].to(device)
    params = batch['params'].to(device)
    out = {}
    for a in a_grid:
        torch.manual_seed(seed)                         # same noise -> isolate a
        a_t = torch.full((cond.shape[0],), float(a), device=device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            g = model.fm.sample(cond, ls, params, n_steps=n_steps, scale_factor=a_t)
        out[a] = _denormalize_to_physical(g.float().cpu().numpy(), ns)
    return out

# Use a fixed set of well-resolved z=0 patches as the structural template.
z0 = z_levels.min()
tmpl = [test_files[i] for i in np.where(zlab == z0)[0]]
train_a = np.array([z_to_a(z) for z in sorted(SNAPSHOT_REDSHIFTS.values())])
fine_a = np.linspace(train_a.min(), train_a.max(), 15)   # interpolated grid
a_grid = np.unique(np.round(np.concatenate([train_a, fine_a]), 4))

swept = repaint_fixed_structure(model, ns, tmpl, a_grid, n_per=64, n_steps=N_STEPS)
z_grid = a_to_z(a_grid)
order = np.argsort(z_grid)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
sweep_panels = [('Gas', 1, False), ('compton_y', 3, True),
                ('temperature', 4, True), ('pressure', 6, True)]
for ax, (nm, ch, th) in zip(axes, sweep_panels):
    amp = []
    for a in a_grid:
        arr = swept[a][:, ch]
        if th:
            amp.append(np.nanmedian([m[m > 0].mean() if np.any(m > 0) else np.nan
                                     for m in arr]))
        else:
            amp.append(np.nanmedian(arr.sum(axis=(-2, -1))))
    amp = np.array(amp)
    ax.plot(z_grid[order], amp[order], 'o-', color='C0')
    for ta in train_a:                                  # mark training redshifts
        ax.axvline(a_to_z(ta), color='grey', ls=':', alpha=.5)
    ax.set_yscale('log'); ax.set_xlabel('conditioning z')
    ax.set_ylabel('median amplitude'); ax.set_title(f'{nm} (fixed structure)')
    ax.grid(alpha=.3)
fig.suptitle('Same DMO structure + params, varying only the conditioning redshift '
             '(dotted = training z)', y=1.03)
plt.tight_layout(); plt.show()
print('If the curves are flat, the conditioning is being ignored; a smooth '
      'monotonic trend through the dotted training-z lines is the goal.')

## 6. Numeric summary

In [ ]:
print('=' * 70)
print(f'{RUN_NAME} / {CKPT_NAME}  (epoch {epoch})  —  {len(real)} test patches')
print(f'redshift levels: {list(np.round(z_levels, 3))}')
print('=' * 70)
ov_mass = [np.median(_rel_mass_err(mass_real[:, c], mass_gen[:, c])) * 100 for c in range(3)]
print('Overall mass |rel err| (%):  '
      + '  '.join(f'{n}={v:.1f}' for n, v in zip(MASS_NAMES, ov_mass)))
for j, nm in enumerate(THERMO_NAMES):
    fl = np.percentile(thermo_real[:, j][thermo_real[:, j] > 0], 50) * 1e-3
    print(f'  {nm:12s} median |Δdex| = {np.median(_dex_err(thermo_real[:, j], thermo_gen[:, j], fl)):.3f}')
print('\nPer-z mass/thermo errors are in §2; amplitude evolution in §3; '
      'Y–M in §4; conditioning response in §5.')
del model; torch.cuda.empty_cache()